# Stage Reach Probability Sandbox

This notebook estimates the probability that each nation reaches each tournament stage.

Interpretation rule:
- stage-reach numbers are the stronger tournament signal because they come from repeated Monte Carlo simulation
- any single displayed scoreline should be treated as a representative scenario, not the main decision signal

Method:
- simulate every group from the current fixtures and current model
- rebuild the official round-of-32 bracket for that simulated table
- let the best-eight-third-place routing create the round-of-32 uncertainty
- simulate the knockout bracket forward once each iteration's path is known

Any matches already locked in the adaptive state store are respected.

In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px

from src.adaptive.state_machine import MatchStateMachine
from src.data.loaders import load_fixtures
from src.models.train import load_or_train_ensemble
import src.simulation.stage_reach as stage_reach_module

stage_reach_module = importlib.reload(stage_reach_module)
build_projected_opponents_table = stage_reach_module.build_projected_opponents_table
build_uncertain_round_of_32_paths = stage_reach_module.build_uncertain_round_of_32_paths
estimate_stage_reach_probabilities = stage_reach_module.estimate_stage_reach_probabilities

pd.options.display.max_columns = 50
pd.options.display.float_format = lambda value: f"{value:.4f}"

In [ ]:
ITERATIONS = 2000
SEED = 42

model = load_or_train_ensemble()
fixtures = load_fixtures()
resolved_results = MatchStateMachine().resolved_results()
teams = sorted(set(fixtures["home_team"]) | set(fixtures["away_team"]))

print(f"Teams: {len(teams)}")
print(f"Group-stage fixtures: {len(fixtures)}")
print(f"Resolved matches already locked: {len(resolved_results)}")

pd.Series(model.weights, name="weight").sort_values(ascending=False)

In [ ]:
result = estimate_stage_reach_probabilities(
    fixtures=fixtures,
    model=model,
    iterations=ITERATIONS,
    seed=SEED,
    resolved_results=resolved_results,
)

stage_probabilities = result.stage_probabilities
round_of_32_opponents = result.round_of_32_opponents
round_of_16_opponents = result.round_of_16_opponents
best_third_group_mix = result.best_third_group_mix

stage_probabilities.head(20)

In [ ]:
stage_columns = [
    "reach_round_of_32",
    "reach_round_of_16",
    "reach_quarter_finals",
    "reach_semi_finals",
    "reach_third_place",
    "reach_final",
    "champion",
]
stage_labels = {
    "reach_round_of_32": "Round of 32",
    "reach_round_of_16": "Round of 16",
    "reach_quarter_finals": "Quarter-finals",
    "reach_semi_finals": "Semi-finals",
    "reach_third_place": "Third-place match",
    "reach_final": "Final",
    "champion": "Champion",
}

top_n_teams = 24
stage_chart_data = (
    stage_probabilities.head(top_n_teams)
    .set_index("team")[stage_columns]
    .rename(columns=stage_labels)
)

fig = px.imshow(
    stage_chart_data,
    aspect="auto",
    color_continuous_scale="YlGnBu",
    labels={"x": "Stage", "y": "Team", "color": "Probability"},
    title=f"Top {top_n_teams} Teams: Monte Carlo Probability to Reach Each Stage",
)
fig.update_layout(height=700)
fig.show()

In [ ]:
r16_heatmap_teams = stage_probabilities.query("reach_round_of_16 > 0").head(16)["team"].tolist()
round_of_16_heatmap = round_of_16_opponents.loc[r16_heatmap_teams, r16_heatmap_teams]

fig = px.imshow(
    round_of_16_heatmap,
    aspect="auto",
    color_continuous_scale="Tealgrn",
    labels={"x": "Possible opponent", "y": "Team", "color": "Probability"},
    title="Round-of-16 Opponent Heatmap from Monte Carlo Qualification Paths",
)
fig.update_layout(height=700)
fig.show()

In [ ]:
uncertain_round_of_32_paths = build_uncertain_round_of_32_paths(
    stage_probabilities,
    round_of_32_opponents,
)

uncertain_round_of_32_paths.head(20)

In [ ]:
best_third_group_mix

In [ ]:
heatmap_teams = stage_probabilities.query("reach_round_of_32 > 0").head(16)["team"].tolist()
round_of_32_heatmap = round_of_32_opponents.loc[heatmap_teams, heatmap_teams]

fig = px.imshow(
    round_of_32_heatmap,
    aspect="auto",
    color_continuous_scale="Sunsetdark",
    labels={"x": "Possible opponent", "y": "Team", "color": "Probability"},
    title="Round-of-32 Opponent Heatmap from Monte Carlo Qualification Paths",
)
fig.update_layout(height=700)
fig.show()

In [ ]:
selected_team = "Argentina"
projected_opponents = build_projected_opponents_table(result, selected_team, top_n_per_stage=8)

projected_opponents.rename(
    columns={
        "stage": "Stage",
        "reach_probability": "Reach Probability",
        "opponent": "Projected Opponent",
        "path_probability": "Path Probability",
        "conditional_opponent_probability": "Opponent Probability | Given Reach",
    }
).assign(
    **{
        "Reach Probability": lambda df: df["Reach Probability"].map(lambda value: f"{value:.1%}"),
        "Path Probability": lambda df: df["Path Probability"].map(lambda value: f"{value:.1%}"),
        "Opponent Probability | Given Reach": lambda df: df["Opponent Probability | Given Reach"].map(lambda value: f"{value:.1%}"),
    }
)

## Notes

- `reach_round_of_32` is the qualification probability from the groups.
- The consolidated projected-opponents table lists possible opponents from the Round of 32 through the Final when that stage is reachable in the simulations.
- `Path Probability` is the unconditional share of all tournament simulations where that exact stage-opponent pairing happens.
- `Opponent Probability | Given Reach` is conditional on the selected team reaching that stage.
- These stage-reach numbers come from repeated tournament simulation, so they are stronger than any one displayed scoreline.
- A representative scoreline and a stage-reach direction can disagree without being a bug: one scoreline is a single scenario, while stage reach sums thousands of plausible paths.
- The round-of-32 opponent matrix is where the best-third-place chaos shows up.
- Once a specific round-of-32 bracket is formed inside an iteration, the later knockout path is structurally fixed again.